In [ ]:
##Dataset  merge and feature engineering

In [ ]:
from config import DATABASE_NAME, BUCKET, REGION, S3_STAGING_DIR

In [ ]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

import sagemaker
from sagemaker.session import Session
from sagemaker.feature_store.feature_group import FeatureGroup
import boto3
import time

region = REGION
role = sagemaker.get_execution_role()
session = sagemaker.Session()

print("Libraries loaded")

In [ ]:
base_path = f"s3://{BUCKET}/raw/corporacion_favorita"

In [ ]:
#Load All 5 Datasets From S3

train = pd.read_csv(f"{base_path}/train/train.csv")
transactions = pd.read_csv(f"{base_path}/transactions/transactions.csv")
stores = pd.read_csv(f"{base_path}/stores/stores.csv")
holidays_events = pd.read_csv(f"{base_path}/holidays/holidays_events.csv")
oil = pd.read_csv(f"{base_path}/oil/oil.csv")

print("All raw datasets loaded successfully")


In [ ]:
#Convert Date Columns

train['date'] = pd.to_datetime(train['date'])
transactions['date'] = pd.to_datetime(transactions['date'])
holidays_events['date'] = pd.to_datetime(holidays_events['date'])
oil['date'] = pd.to_datetime(oil['date'])


In [ ]:
"

| Dataset         | Join Key        |
| --------------- | --------------- |
| train           | date, store_nbr |
| transactions    | date, store_nbr |
| stores          | store_nbr       |
| holidays_events | date            |
| oil             | date            |


In [ ]:
#Start With Train as Base

df = train.copy()
print(df.shape)

In [ ]:
#Merge Transactions data with train

'''Merge on:
date
store_nbr'''

df = df.merge(
    transactions,
    on=['date', 'store_nbr'],
    how='left'
)

print("After merging transactions:", df.shape)


In [ ]:
#Merge Stores
'''
Merge on:
store_nbr'''

df = df.merge(
    stores,
    on='store_nbr',
    how='left'
)

print("After merging stores:", df.shape)


In [ ]:
#Merge Oil
'''
Merge on:
date'''

df = df.merge(
    oil,
    on='date',
    how='left'
)

print("After merging oil:", df.shape)


In [ ]:
df['dcoilwtico'] = df['dcoilwtico'].ffill()

In [ ]:
df.columns

In [ ]:
#Merge Holidays_events

In [ ]:
holidays_events.groupby("date").size().sort_values(ascending=False).head(10)

In [ ]:
"
What This Means

For example:

📅 2014-06-25 → 4 rows in holidays table

That means on that date there are 4 different holiday/event records.

When you merge with train:

If train has 10,000 rows for that date
After merge → 10,000 × 4 = 40,000 rows

That is exactly why your dataset increased by 7,392 rows.
This is a many-to-one merge becoming many-to-many.

Why June 25 Appears So Often?

In Ecuador:
Some holidays are regional
Some are local
Some are events
Some are transferred
So one calendar day can have:
1 national holiday
1 regional holiday
1 local holiday
1 event

= 4 rows for same date

In [ ]:
df.shape

In [ ]:
#Prepare Holidays Data

hol = holidays_events.copy()
# Remove transferred holidays
hol = hol[hol["transferred"] == False]

In [ ]:
#NATIONAL Holidays

national = hol[hol["locale"] == "National"].copy()

national["is_national"] = 1

national = national.groupby("date", as_index=False).agg({
    "is_national": "max"
})

# Merge
df = df.merge(national, on="date", how="left")

df["is_national"] = df["is_national"].fillna(0)


In [ ]:
#REGIONAL Holidays

regional = hol[hol["locale"] == "Regional"].copy()

regional["is_regional"] = 1

regional = regional.groupby(
    ["date", "locale_name"],
    as_index=False
).agg({
    "is_regional": "max"
})

df = df.merge(
    regional,
    left_on=["date", "state"],
    right_on=["date", "locale_name"],
    how="left"
)

df.drop("locale_name", axis=1, inplace=True)

df["is_regional"] = df["is_regional"].fillna(0)


In [ ]:
#LOCAL Holidays

local = hol[hol["locale"] == "Local"].copy()

local["is_local"] = 1

local = local.groupby(
    ["date", "locale_name"],
    as_index=False
).agg({
    "is_local": "max"
})

df = df.merge(
    local,
    left_on=["date", "city"],
    right_on=["date", "locale_name"],
    how="left"
)

df.drop("locale_name", axis=1, inplace=True)

df["is_local"] = df["is_local"].fillna(0)


In [ ]:
"
"For every row in df, check:

Does the date match?

Does the state match locale_name?
If yes → attach holiday info"

What Happens Internally?abs

In df:
| date       | store_nbr | state     |
| ---------- | --------- | --------- |
| 2013-07-25 | 5         | Pichincha |
| 2013-07-25 | 8         | Guayas    |

In holidays (regional):
| date       | locale_name | is_regional |
| ---------- | ----------- | ----------- |
| 2013-07-25 | Pichincha   | 1           |

After merge:
| date       | store_nbr | state     | is_regional |
| ---------- | --------- | --------- | ----------- |
| 2013-07-25 | 5         | Pichincha | 1           |
| 2013-07-25 | 8         | Guayas    | NaN         

df["is_regional"] = df["is_regional"].fillna(0)
| state     | is_regional |
| --------- | ----------- |
| Pichincha | 1           |
| Guayas    | 0           |


In [ ]:
df.rename(columns={"type": "store_type"}, inplace=True)

In [ ]:
df.head()

In [ ]:
print(df.shape)

In [ ]:
#store the merged data into train_merged.csv and store in s3 bucket
df.to_csv(
    f"s3://{BUCKET}/processed/merged/train_merged.csv",
    index=False
)

In [ ]:
##Feature Engineering 

In [ ]:
#read the stored data
df = pd.read_csv(
    f"s3://{BUCKET}/processed/merged/train_merged.csv",
    parse_dates=['date']
)


In [ ]:
#list all the columns present in the data
df.columns

In [ ]:
#Check the information regarding the data
df.info()

In [ ]:
#check for null values
df.isna().sum()

In [ ]:
#sort the data via store_nbr,family and date
df = df.sort_values(["store_nbr", "family", "date"])
df.head()

In [ ]:
"
1) Why are transactions NaN?

Important concept:
Your dataset is at:
store_nbr + family + date
But transactions dataset is at:
store_nbr + date

So when you merged:

For some dates, transactions don’t exist
Example: 2013-01-01 (holiday — store closed)
So merge gives NaN
This is expected behavior.

Logical reasoning:

If there are no transactions recorded →
It likely means store was closed → so transactions = 0

So safest fix:

df["transactions"] = df["transactions"].fillna(0)
✔ This is correct for retail forecasting.


2) Why is dcoilwtico NaN?
Oil prices:
Some dates missing
Possibly weekends or missing reporting days
Oil price is continuous economic variable, so:

🚫 Do NOT replace with 0
That would distort model.

Correct Way to Handle Oil Price

Use forward fill (carry last known value):

df["dcoilwtico"] = df["dcoilwtico"].fillna(method="ffill")

Then if still any NaNs at beginning:

df["dcoilwtico"] = df["dcoilwtico"].fillna(method="bfill")
✔ This preserves economic trend.

In [ ]:
# Fill transactions NaN
df["transactions"] = df["transactions"].fillna(0)

# Fill oil prices NaN
df["dcoilwtico"] = df["dcoilwtico"].fillna(method="ffill")
df["dcoilwtico"] = df["dcoilwtico"].fillna(method="bfill")

# Final check
print(df.isna().sum())

In [ ]:
#Time Features (Mandatory)

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["dayofweek"] = df["date"].dt.dayofweek
df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)

df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
df["is_month_end"] = df["date"].dt.is_month_end.astype(int)

In [ ]:
"
These capture:

Weekly seasonality
Monthly salary effect (15th, month-end)
Weekend patterns

In [ ]:
#Lag Features(Sales)

lags = [1, 7, 14, 28]

for lag in lags:
    df[f"sales_lag_{lag}"] = (
        df.groupby(["store_nbr", "family"])["sales"]
        .shift(lag)
    )

In [ ]:
"
Why?

Lag 1 → yesterday effect
Lag 7 → weekly pattern
Lag 14 → bi-weekly salary cycle
Lag 28 → monthly pattern
These directly capture patterns you saw in ACF.

In [ ]:
#Rolling Mean Features(Sales)

windows = [7, 14, 30]

for window in windows:
    df[f"rolling_mean_{window}"] = (
        df.groupby(["store_nbr", "family"])["sales"]
        .shift(1)
        .rolling(window)
        .mean()
    )


In [ ]:
#Promotion Features-Captures promotion intensity trend.

df["promo_last_7"] = (
    df.groupby(["store_nbr", "family"])["onpromotion"]
    .shift(1)
    .rolling(7)
    .sum()
)

df["promo_last_14"] = (
    df.groupby(["store_nbr", "family"])["onpromotion"]
    .shift(1)
    .rolling(14)
    .sum()
)

In [ ]:
#Create lag features for transactions column

lags = [7, 14, 30]

for lag in lags:
    df[f"trans_lag_{lag}"] = (
        df.groupby("store_nbr")["transactions"]
          .shift(lag)
    )


In [ ]:
#Create rolling mean features for transactions column

windows = [7, 14, 30]

for window in windows:
    df[f"trans_roll_{window}"] = (
        df.groupby("store_nbr")["transactions"]
          .shift(1)
          .rolling(window)
          .mean()
    )

In [ ]:
#Holiday Interaction Features

df["holiday_promo"] = df["is_national"] * df["onpromotion"]
df["holiday_weekend"] = df["is_national"] * df["is_weekend"]

In [ ]:
#Holiday Season Feature

df["is_holiday_season"] = (
    (df["month"].isin([11, 12])) | 
    (df["is_national"] == 1)
).astype(int)

In [ ]:
"
From your EDA:

December spike
Likely driven by Christmas + year-end spending
November may include early holiday buildup

In [ ]:
#Cyclical Encoding
import numpy as np

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)


In [ ]:
"
What This Does

It converts month into coordinates on a circle:
month_sin → vertical position
month_cos → horizontal position
Now:

December and January become close in value
March and April become close

Opposite months become opposite points
The model now understands seasonality smoothly.


In [ ]:
#Oil Price Features

df["oil_lag_7"] = df["dcoilwtico"].shift(7)
df["oil_rolling_30"] = df["dcoilwtico"].rolling(30).mean()

In [ ]:
#Cyclic encoding for week

df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

In [ ]:
#Interaction Feature (December + Friday)

df['december_friday'] = ((df['month']==12) & (df['dayofweek']==4)).astype(int)

In [ ]:
"
From EDA:

December = spike
Friday = high sales
Combination may explode sales
That’s interaction modeling.

In [ ]:
#Earthquake Feature

df['is_earthquake_period'] = (
    (df['date'] >= '2016-04-16') & 
    (df['date'] <= '2016-05-31')
).astype(int)

In [ ]:
#Payroll Feature (VERY IMPORTANT)
'''
Public wages paid:
15th
Last day of month'''

df["is_payday"] = (
    (df["date"].dt.day == 15) |
    (df["date"].dt.is_month_end)
).astype(int)

In [ ]:
df.isna().sum()

In [ ]:
df = df.dropna()

In [ ]:
df = df.rename(columns={
    "rolling_mean_7": "sales_rolling_mean_7",
    "rolling_mean_14 ": "sales_rolling_mean_14 ",
    "rolling_mean_30": "sales_rolling_mean_30"
})

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
#EDA during Earthquake period(2016-04-16 to 2016-05-31)

In [ ]:
quake_df = df[
    (df['date'] >= '2016-04-16') & 
    (df['date'] <= '2016-05-31')
].copy()

print("Rows in earthquake period:", quake_df.shape[0])

In [ ]:
#Daily Total Sales During Earthquake
import matplotlib.pyplot as plt
daily_quake = quake_df.groupby('date').agg({
    'sales': 'sum',
    'transactions': 'sum'
}).reset_index()

plt.figure(figsize=(14,5))
plt.plot(daily_quake['date'], daily_quake['sales'], color='darkred')
plt.title("Daily Sales During Earthquake Period")
plt.xlabel("Date")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.show()


In [ ]:
#Daily Transactions During Earthquake

plt.figure(figsize=(14,5))
plt.plot(daily_quake['date'], daily_quake['transactions'], color='darkblue')
plt.title("Daily Transactions During Earthquake Period")
plt.xlabel("Date")
plt.ylabel("Total Transactions")
plt.xticks(rotation=45)
plt.show()

In [ ]:
family_spike = quake_df.groupby('family')['sales'].sum().sort_values(ascending=False)

family_spike.head(10)


In [ ]:
daily_quake = quake_df.groupby('date').agg({
    'sales': 'sum',
    'transactions': 'sum'
}).reset_index()

import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(14,6))

# Sales (Left Y-axis)
ax1.plot(daily_quake['date'], 
         daily_quake['sales'], 
         color='darkred',
         label='Sales')
ax1.set_xlabel('Date')
ax1.set_ylabel('Total Sales', color='darkred')

# Transactions (Right Y-axis)
ax2 = ax1.twinx()
ax2.plot(daily_quake['date'], 
         daily_quake['transactions'], 
         color='navy',
         label='Transactions')
ax2.set_ylabel('Total Transactions', color='navy')

# Earthquake vertical line
plt.axvline(pd.to_datetime('2016-04-16'),
            color='black',
            linestyle='--',
            label='Earthquake Date')

plt.title('Sales and Transactions During Earthquake Period')
fig.tight_layout()
plt.show()


In [ ]:
#30-Day Before vs After Earthquake Analysis

import pandas as pd
import matplotlib.pyplot as plt

# Ensure datetime
df['date'] = pd.to_datetime(df['date'])

earthquake_date = pd.to_datetime("2016-04-16")

# Define periods
before_period = df[(df['date'] >= earthquake_date - pd.Timedelta(days=30)) & 
                   (df['date'] < earthquake_date)]

after_period = df[(df['date'] > earthquake_date) & 
                  (df['date'] <= earthquake_date + pd.Timedelta(days=30))]

# ---- Compute averages ----
avg_sales_before = before_period['sales'].mean()
avg_sales_after = after_period['sales'].mean()

percent_change = ((avg_sales_after - avg_sales_before) / avg_sales_before) * 100

print("Average Sales 30 Days BEFORE Earthquake:", round(avg_sales_before, 2))
print("Average Sales 30 Days AFTER Earthquake:", round(avg_sales_after, 2))
print("Percentage Change:", round(percent_change, 2), "%")


# ---- Daily aggregation for smoother visualization ----
daily = df.groupby('date')['sales'].sum().reset_index()

window = daily[(daily['date'] >= earthquake_date - pd.Timedelta(days=30)) &
               (daily['date'] <= earthquake_date + pd.Timedelta(days=30))]

# Use rolling mean (7-day)
window['rolling_mean_7'] = window['sales'].rolling(7).mean()

# ---- Plot ----
plt.figure()
plt.plot(window['date'], window['rolling_mean_7'])
plt.axvline(earthquake_date)
plt.title("7-Day Rolling Mean Sales (30 Days Before & After Earthquake)")
plt.xlabel("Date")
plt.ylabel("Rolling Mean Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



In [ ]:
#Count how many holiday days during earthquake

# Define earthquake period
start = '2016-04-16'
end = '2016-05-31'

df_eq = df[
    (df['date'] >= start) &
    (df['date'] <= end)
]

# Count how many holiday days
df_eq.groupby('date')[['is_national','is_regional','is_local']].max().reset_index()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ================================
# Define Period (30 days before & after earthquake)
# ================================
start = '2016-03-17'
end   = '2016-05-31'

df_period = df[
    (df['date'] >= start) &
    (df['date'] <= end)
].copy()

# ================================
# Get Holiday Dates
# ================================
holiday_dates = df_period[
    (df_period['is_national'] == 1) |
    (df_period['is_regional'] == 1) |
    (df_period['is_local'] == 1)
]['date'].unique()

# ================================
# Aggregate Daily Rolling Mean
# (Important because multiple rows per day)
# ================================
daily = df_period.groupby('date').agg({
    'sales_rolling_mean_7': 'mean'
}).reset_index()

# ================================
# Plot
# ================================
plt.figure(figsize=(14,6))

plt.plot(daily['date'], daily['sales_rolling_mean_7'], 
         label='7-Day Rolling Mean Sales')

# Mark holidays
for h in holiday_dates:
    plt.axvline(h, linestyle='--', alpha=0.4)

# Mark earthquake date
plt.axvline(pd.to_datetime('2016-04-16'), 
            color='red', 
            linewidth=2, 
            label='Earthquake')

plt.title('7-Day Rolling Mean Sales with Holidays (Earthquake Period)')
plt.xlabel('Date')
plt.ylabel('Rolling Mean Sales')
plt.legend()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Ensure date is datetime
df['date'] = pd.to_datetime(df['date'])

# Earthquake window
start = '2016-04-16'
end = '2016-05-31'

eq_df = df[(df['date'] >= start) & (df['date'] <= end)].copy()

# Aggregate by date
daily_eq = eq_df.groupby('date').agg({
    'sales': 'sum',
    'onpromotion': 'sum'
}).reset_index()

# Promotion flag
daily_eq['has_promo'] = (daily_eq['onpromotion'] > 0).astype(int)

# Average sales comparison
promo_sales = daily_eq[daily_eq['has_promo']==1]['sales'].mean()
no_promo_sales = daily_eq[daily_eq['has_promo']==0]['sales'].mean()

print("Average Sales WITH Promotion:", promo_sales)
print("Average Sales WITHOUT Promotion:", no_promo_sales)

# Plot
plt.figure(figsize=(12,6))
plt.plot(daily_eq['date'], daily_eq['sales'], label='Total Sales')
plt.plot(daily_eq['date'], daily_eq['onpromotion'], label='Total Promotions')

plt.axvline(pd.to_datetime('2016-04-16'), color='red', linewidth=2, label='Earthquake')

plt.title("Sales vs Promotions During Earthquake Period")
plt.legend()
plt.show()

In [ ]:
correlation = daily_eq['sales'].corr(daily_eq['onpromotion'])
print("Correlation between Sales and Promotions:", correlation)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Earthquake period window (30 days before & after optional)
start_date = '2016-03-17'
end_date   = '2016-05-31'

earthquake_date = pd.to_datetime('2016-04-16')

# Filter window
oil_eq = df[(df['date'] >= start_date) & (df['date'] <= end_date)]

# Aggregate daily (important if multiple stores)
oil_daily = oil_eq.groupby('date').agg({
    'oil_rolling_30': 'mean'
}).reset_index()

# Plot
plt.figure(figsize=(12,6))
plt.plot(oil_daily['date'],
         oil_daily['oil_rolling_30'])

plt.axvline(earthquake_date, linewidth=2)

plt.title('30-Day Rolling Mean Oil Price During Earthquake Period')
plt.xlabel('Date')
plt.ylabel('Oil Price (Rolling Mean)')
plt.xticks(rotation=45)
plt.show()

In [ ]:
#Create Feature store

In [ ]:
#Prepare Data
'''
Feature Store requires:
Record identifier column
Event time column as string
'''

df['store_family_id'] = (
    df['store_nbr'].astype(str) + "_" + df['family']
)


# Convert datetime columns
df['date'] = df['date'].astype(str)
df['event_time'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Double check
print(df.dtypes)

In [ ]:
df['event_time'].isna().sum()

In [ ]:
# Create feature group object
feature_group = FeatureGroup(
    name="store-sales-feature-group_version1",
    sagemaker_session=session
)

# Load feature definitions
feature_group.load_feature_definitions(data_frame=df)

# Create feature group
feature_group.create(
    s3_uri=f"s3://{BUCKET}/feature-store/",
    record_identifier_name="store_family_id",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=True
)

In [ ]:
feature_group.describe()

In [ ]:
feature_group.ingest(data_frame=df, max_workers=3, wait=True)

In [ ]:
"
This will:

Store data in Offline Store (S3)
Store data in Online Store
Create Athena table automatically

In [ ]:
"
We now have:

- A registered feature group
- Schema stored
- Data versioned
- Features queryable
- Production ready

This is enterprise-level ML architecture

In [ ]:
#save the file offline
df.to_csv("processed_features.csv", index=False)

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualization (optional but useful)
import matplotlib.pyplot as plt
import seaborn as sns

# Model
import xgboost as xgb

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

import sagemaker
from sagemaker.session import Session
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker import get_execution_role

import boto3
from config import *
import joblib

In [ ]:
feature_group.describe()

In [ ]:
database = feature_group.describe()['OfflineStoreConfig']['DataCatalogConfig']['Database']
table = feature_group.describe()['OfflineStoreConfig']['DataCatalogConfig']['TableName']

In [ ]:
athena_query = f"""
SELECT *
FROM "{database}"."{table}"
"""

query = feature_group.athena_query()
query.run(
    query_string=athena_query,
    output_location=f"s3://{bucket}/athena-output/"
)

df = query.as_dataframe()

#Now df contains your processed features from Feature Store.

In [ ]:
'''
Feature Store automatically adds:

api_invocation_time

write_time

is_deleted

Drop them:
'''

df = df.drop(columns=["api_invocation_time", "write_time", "is_deleted"], errors="ignore")

'''
Now use df normally for:

Train/val/test split

Model training
'''

In [ ]:
'''Online store is used for:

Real-time feature lookup

Endpoint predictions

This retrieves latest features for a store-family combination.

This is used when:

You deploy endpoint

You want live predictions
'''

response = feature_group.get_record(
    record_identifier_value="1_BEVERAGES"
)

response

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

print(df["date"].min(), df["date"].max())
print(df.shape)